# Data Science · Woche 4 · Praktikum

**B.Sc. Wirtschaftsinformatik · Hochschule Ruhr West · Do 29.10.2026, 16:55–17:45**

Heute stellen Sie die **vier Fragen aus der Vorlesung** selbst: erst an eine kleine Umfrage über Katzen, dann an die
Olist-Daten. Am Ende steht ein kurzer Qualitätsbericht – genau das, was Sie für Ihren Projektdatensatz bis zum ersten
Zwischenstand am 03.12. brauchen.

| Zeit | Was |
|---|---|
| 16:55 | Gemeinsam am Beamer: Was heute geübt wird – und wie eine Aufgabe gebaut ist (5 Minuten) |
| 17:00 | Zu zweit: **Teil 1 · Katzen** (Frage 2) · **Teil 2 · Steckbrief** (Fragen 1 und 2) · **Teil 3 · Lieferzeiten** (Frage 3) · **Teil 4 · Qualität** (Frage 4) |
| 17:30 | Jedes Team stellt zwei Zeilen seines Qualitätsberichts vor (2 Minuten, ganz unten vor Teil 5) |
| 17:45 | Schluss · **Teil 5 · Ihr Projekt** machen Sie zu Hause |

## So ist jede Aufgabe gebaut (wie in Woche 2 und 3)

1. **Vorgemacht** – fertiger Code mit Kommentaren: ausführen (`Shift + Enter`) und lesen. Wo **🔮 Vorhersage** steht,
   schreiben Sie zuerst Ihre Vermutung auf und führen dann aus.
2. **Jetzt Sie** – dasselbe noch einmal, mit anderen Spalten. Sie ersetzen nur das `None` hinter `# TODO`.
3. **Check** – darunter steht ✓ richtig · ✗ prüfen (mit dem erwarteten Wert) · … noch offen.
4. **Hilfe in zwei Stufen:** erst **💡 Tipp** (wo das Muster steht), dann **🧩 Gerüst** (Ihre Zeilen mit Lücken `___`).
   Hängen Sie an der Schreibweise, öffnet **📘** den passenden Abschnitt im Grundlagen-Heft von Woche 2.

**Pflicht** sind acht Aufgaben – **1.4 · 2.3 · 2.4 · 2.5 · 3.2 · 3.3 · 4.2 · 4.5** – und der Bericht **4.7**. Alles mit
**⭐** ist für alle, die schneller sind. Ganz unten steht die **Kür**: die Wetterstation Essen-Bredeney aus der
Vorlesung selbst prüfen – mit echten Daten des Deutschen Wetterdienstes. *Um 17:25 springen alle zu 4.7.*

## Spickzettel: die vier Fragen in Python

| Frage | Python |
|---|---|
| 1 · Woran wurde gemessen? | `df.shape` · `df['id'].is_unique` · `df['id'].nunique()` · `df.groupby(tag).size()` |
| 2 · Was wurde gemessen? | `df.dtypes` · `pd.to_numeric(spalte, errors='coerce')` |
| 3 · Was ist herausgekommen? | `spalte.describe()` · `spalte.quantile(0.9)` · `df.boxplot(column=…, by=…)` |
| 4 · Wie gut ist die Messung? | `df.isna().sum()` · `df.duplicated().sum()` · `(df['a'] < df['b']).sum()` |

## 0 · Setup (1 Minute)

**Datei → Kopie in Drive speichern**, dann die Zelle unten ausführen. Sie holt drei Olist-Tabellen und die Übersetzung
der Produktkategorien aus dem Kurs-Repository. Erwartet: `orders: 99441 · kunden: 99441 · produkte: 32951` und
darunter `Setup ok`.

*Falls das Laden scheitert:* aus https://github.com/ZhuosHub/Data_Science_HRW_Praktikum (Ordner `data/olist`) die Dateien herunterladen, links über das
Ordner-Symbol hochladen und die Zelle erneut ausführen.

In [ ]:
import numpy as np                  # Zahlen
import pandas as pd                 # Tabellen
import matplotlib.pyplot as plt     # Diagramme

REPO_URL = 'https://raw.githubusercontent.com/ZhuosHub/Data_Science_HRW_Praktikum/main/data/'
KATZEN_URL = 'https://raw.githubusercontent.com/tidyverse/datascience-box/main/course-materials/_slides/u2-d10-data-types/data/cat-lovers.csv'


def lade(name, **kw):
    """Sucht die Datei im Repository-Ordner, dann auf GitHub, dann im Arbeitsordner (hochgeladen)."""
    for quelle in ['../data/' + name, REPO_URL + name, name.split('/')[-1]]:
        try:
            return pd.read_csv(quelle, **kw)
        except (FileNotFoundError, OSError) as fehler:
            if quelle.startswith('http'):
                print('GitHub nicht erreichbar:', fehler)
    raise FileNotFoundError(f'{name} nicht gefunden – siehe „Falls das Laden scheitert“ über dieser Zelle')


zeiten = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
          'order_delivered_customer_date', 'order_estimated_delivery_date']
orders = lade('olist/olist_orders_dataset.csv', parse_dates=zeiten)          # parse_dates: diese Spalten sind Zeitpunkte
kunden = lade('olist/olist_customers_dataset.csv')
produkte = lade('olist/olist_products_dataset.csv')
uebersetzung = lade('olist/product_category_name_translation.csv')          # Kategorie portugiesisch → englisch
print(f'orders: {len(orders)} · kunden: {len(kunden)} · produkte: {len(produkte)}')
print('Setup ok')

---
# Teil 1 · Wie viele Katzen haben Sie? (zu zweit, ca. 7 Minuten)

Aus der Vorlesung: In einer Umfrage aus „Data Science in a Box“ sollten 60 Menschen ihre Zahl an Katzen **als Zahl**
eingeben. Zwei haben es nicht getan. Hier sehen Sie, was das in pandas anrichtet – und wie man die Übeltäter findet
und repariert. Das ist Frage 2 der Vorlesung: Passt der Datentyp zur Bedeutung?

> 📘 Hängen Sie an der Schreibweise? Ein Klick öffnet das Grundlagen-Heft aus Woche 2 in einem neuen Tab an der richtigen Stelle: [Abschnitt 5 · pandas: Spalten und Tabellen](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-5) und [Abschnitt 10 · Fehlermeldungen lesen](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-10).

In [ ]:
# ===== Vorgemacht (1.1): die Umfrage laden – Frage 1 und 2 =====
try:
    katzen = pd.read_csv(KATZEN_URL)                     # direkt von GitHub (Data Science in a Box)
except (OSError, ValueError):
    katzen = lade('katzen/cat-lovers.csv')               # Ersatz: aus dem Kurs-Repository oder hochgeladen
print(katzen.shape)                                      # Frage 1: Zeilen × Spalten – eine Zeile = eine Person
print(katzen.dtypes)                                     # Frage 2: der Datentyp je Spalte
katzen.head()

`number_of_cats` ist **Text** (`object`, in pandas 3 heißt das `str`), obwohl nach einer Zahl gefragt wurde.

## 1.2 · 🔮 Vorhersage: der Mittelwert

Was kommt bei `katzen['number_of_cats'].mean()` heraus – etwa 1 Katze? Doppelklick, Vermutung eintragen, dann
ausführen.

*Meine Vorhersage:* …

In [ ]:
# ===== Vorgemacht (1.2): der Mittelwert einer Text-Spalte =====
try:
    print(katzen['number_of_cats'].mean())
except TypeError as fehler:              # try/except fängt den Fehler ab, damit „Alle ausführen“ weiterläuft
    print('TypeError:', str(fehler)[:150])

Eine Fehlermeldung: Von Text gibt es keinen Mittelwert. **Ein einziger Text-Eintrag macht die ganze Spalte zu Text.**

## 1.3 · Die Übeltäter finden

`pd.to_numeric(…, errors='coerce')` wandelt um, was eine Zahl ist. Alles andere wird zu `NaN` (*not a number*, ein
fehlender Wert) – und genau diese Zeilen suchen wir.

In [ ]:
# ===== Vorgemacht (1.3): was ist keine Zahl? =====
zahl = pd.to_numeric(katzen['number_of_cats'], errors='coerce')     # Zahl oder NaN
print('keine Zahl:', katzen.loc[zahl.isna(), 'number_of_cats'].tolist())
print('Mittelwert ohne die beiden:', round(zahl.mean(), 3))          # mean() lässt NaN einfach weg

## 1.4 · Jetzt Sie: reparieren und rechnen

Aus `three` wird 3 – das ist klar. Bei der halb menschlichen Katze müssen **Sie entscheiden**: 1,5 oder 2?

In [ ]:
# ===== Jetzt Sie (1.4): reparieren, dann rechnen =====
zahl_repariert = zahl.copy()                                                    # fertig: eine Kopie zum Reparieren
zahl_repariert[katzen['number_of_cats'] == 'three'] = 3                         # fertig: „three“ ist 3
zahl_repariert[katzen['number_of_cats'].astype(str).str.startswith('1.5')] = None   # TODO: 1.5 oder 2? Zahl statt None
mittel_katzen = None                                                            # TODO: der Mittelwert von zahl_repariert

# --- Check ---
if mittel_katzen is None:
    print('… noch offen')
elif zahl_repariert.isna().any():
    print('✗ prüfen: es ist noch ein NaN übrig – haben Sie bei der halben Katze eine Zahl statt None eingetragen?')
elif abs(mittel_katzen - zahl_repariert.mean()) > 1e-9:
    print('✗ prüfen: erwartet zahl_repariert.mean()')
elif round(mittel_katzen, 3) == 0.833:
    print('✓ richtig · 0,833 Katzen je Person – wie die Kursleitung, die die halbe Katze als 2 gezählt hat.')
else:
    print(f'✓ richtig gerechnet · {mittel_katzen:.3f} Katzen je Person. Die Kursleitung hat 2 gewählt und kommt '
          'auf 0,833 – beides ist begründbar, wenn Sie Ihre Entscheidung dazuschreiben.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>In der Zeile darüber steht das Muster: links die Zeilen, die ersetzt werden, rechts der neue Wert. Ersetzen Sie <code>None</code> durch <code>1.5</code> oder <code>2</code> (Dezimalpunkt!). Der Mittelwert ist <code>.mean()</code> an <code>zahl_repariert</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>zahl_repariert[katzen[&#x27;number_of_cats&#x27;].astype(str).str.startswith(&#x27;1.5&#x27;)] = ___
mittel_katzen = zahl_repariert.___()</pre>
</details>

**1.5 Deuten (ein Satz):** Warum ist „halbe Katze = 2“ eine Entscheidung und kein Messwert – und wo schreiben Sie so eine Entscheidung hin?

*Ihre Antwort:*

---
# Teil 2 · Steckbrief: Woran und was wurde gemessen? (zu zweit, ca. 10 Minuten)

Frage 1 und 2 der Vorlesung an die Olist-Tabellen. Frage 1: Wofür steht eine Zeile, und welcher Schlüssel kennzeichnet
sie? Frage 2: Was steht in den Spalten, und passt der Datentyp zur Bedeutung? Ein **Steckbrief** beantwortet das für
alle Spalten auf einmal.

> 📘 Hängen Sie an der Schreibweise? Ein Klick öffnet das Grundlagen-Heft aus Woche 2 in einem neuen Tab an der richtigen Stelle: [Abschnitt 5 · pandas: Spalten und Tabellen](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-5) und [Abschnitt 8 · Schleife und Dictionary](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-8).

In [ ]:
# ===== Vorgemacht (2.1): Frage 1 an orders – Zeile und Schlüssel =====
print(orders.shape)                         # Zeilen × Spalten: eine Zeile = eine Bestellung
print(orders['order_id'].is_unique)         # True: jede order_id kommt genau einmal vor – ein Schlüssel

In [ ]:
# ===== Vorgemacht (2.2): eine Funktion für den Steckbrief – eine Zeile je Spalte =====
def steckbrief(df):
    """Je Spalte: Datentyp, fehlende Werte (Anzahl und Prozent), Zahl verschiedener Werte, ein Beispiel."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),                          # Frage 2: wie gespeichert?
        'fehlend': df.isna().sum(),                              # Frage 4: Vollständigkeit
        'fehlend_%': (df.isna().mean() * 100).round(1),
        'verschieden': df.nunique(),                             # 2 Werte? 10? so viele wie Zeilen (Schlüssel)?
        'beispiel': df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })


steckbrief(orders)

In `orders` fehlen Zeitpunkte – bei Bestellungen, die (noch) nicht freigegeben, verschickt oder zugestellt wurden.
`order_id` hat so viele verschiedene Werte wie Zeilen: der Schlüssel.

## 2.3 · Jetzt Sie: der Schlüssel der Kunden

Wie bei Hubway in der Vorlesung: Eine Zeile ist nicht automatisch ein Mensch. Olist hat 99 441 `customer_id`, aber nur
96 096 echte Menschen. Rechnen Sie es nach.

In [ ]:
# ===== Jetzt Sie (2.3): Schlüssel der Tabelle kunden =====
id_eindeutig = None      # TODO: Ist customer_id ein Schlüssel? True oder False
menschen = None          # TODO: Wie viele verschiedene customer_unique_id gibt es?

# --- Check ---
if id_eindeutig is None or menschen is None:
    print('… noch offen')
elif id_eindeutig is not True:
    print('✗ prüfen: erwartet True – kunden[\'customer_id\'].is_unique')
elif menschen != 96096:
    print('✗ prüfen: erwartet 96 096 – kunden[\'customer_unique_id\'].nunique()')
else:
    print(f'✓ richtig · customer_id ist eindeutig, aber nur {menschen} Menschen: Wer wieder bestellt, bekommt eine '
          'neue customer_id. Eindeutige ID ≠ eindeutiger Mensch.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Wie in 2.1 bei <code>orders['order_id']</code>: Spalte auswählen, dann <code>.is_unique</code> (ohne Klammern). Verschiedene Werte zählt <code>.nunique()</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>id_eindeutig = kunden[&#x27;customer_id&#x27;].___
menschen = kunden[&#x27;customer_unique_id&#x27;].___()</pre>
</details>

## 2.4 · Jetzt Sie: der Steckbrief der Produkte

In [ ]:
# ===== Jetzt Sie (2.4): Steckbrief der Produkte – und eine Zahl daraus ablesen =====
sb_produkte = None       # TODO: der Steckbrief der Tabelle produkte
ohne_kategorie = None    # TODO: aus sb_produkte: Zeile 'product_category_name', Spalte 'fehlend'

# --- Check ---
if sb_produkte is None or ohne_kategorie is None:
    print('… noch offen')
elif ohne_kategorie != 610:
    print('✗ prüfen: erwartet 610 – sb_produkte.loc[\'product_category_name\', \'fehlend\']')
else:
    print(f'✓ richtig · {ohne_kategorie} Produkte ohne Kategorie. Schauen Sie in den Steckbrief: Bei genau so vielen '
          'fehlen auch Namenslänge, Beschreibung und Fotozahl.')
    print(sb_produkte)

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Die Funktion aus 2.2 aufrufen, diesmal mit <code>produkte</code> in der Klammer. Einen einzelnen Wert holen Sie mit <code>.loc[zeile, spalte]</code> – wie in Woche 3 bei <code>m.loc['mit', 'sem']</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>sb_produkte = steckbrief(___)
ohne_kategorie = sb_produkte.loc[&#x27;product_category_name&#x27;, &#x27;___&#x27;]</pre>
</details>

## 2.5 · Jetzt Sie: Granularität und Umfang – eine Zeile pro Tag

Aus der Vorlesung: Eine Zeile in `orders` ist eine Bestellung, Schlüssel `order_id`. Für die Frage „Wie viele
Bestellungen gehen pro Tag ein?“ brauchen Sie eine gröbere Tabelle: eine Zeile pro Tag – anderer Schlüssel, andere
Beobachtungseinheit. In pandas: den Tag aus dem Zeitstempel ziehen und mit `groupby` zusammenfassen. Und der Umfang:
Ab wann gibt es überhaupt Bestellungen?

In [ ]:
# ===== Jetzt Sie (2.5): eine Zeile pro Tag – und der Umfang =====
tag = orders['order_purchase_timestamp'].dt.date     # fertig: der Tag aus dem Zeitstempel
pro_tag = None           # TODO: Bestellungen je Tag – orders.groupby(tag).size()
spitzentag = None        # TODO: der Tag mit den meisten Bestellungen – pro_tag.idxmax()
erste = None             # TODO: der früheste Bestellzeitpunkt – .min() der Spalte order_purchase_timestamp

# --- Check ---
if pro_tag is None or spitzentag is None or erste is None:
    print('… noch offen')
elif len(pro_tag) != 634:
    print('✗ prüfen: erwartet 634 Tage – orders.groupby(tag).size()')
elif str(spitzentag) != '2017-11-24':
    print('✗ prüfen: erwartet 2017-11-24 – pro_tag.idxmax()')
elif str(erste) != '2016-09-04 21:15:19':
    print('✗ prüfen: erwartet 2016-09-04 21:15:19 – orders[\'order_purchase_timestamp\'].min()')
else:
    bis = orders['order_purchase_timestamp'].max().date()
    print(f'✓ richtig · {len(pro_tag)} Tage, eine Zeile pro Tag. Spitzentag {spitzentag} mit {pro_tag.max()} '
          f'Bestellungen – der Black Friday 2017. Umfang: {erste.date()} bis {bis}. Wer bei Olist nie bestellt hat, '
          'steht in keiner Zeile – das sagen nur die Metadaten (die Beschreibung des Datensatzes bei Kaggle).')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Wie auf der Folie: <code>orders.groupby(tag).size()</code> zählt die Zeilen je Tag. Den größten Wert findet <code>.max()</code>, seinen Index – den Tag – <code>.idxmax()</code>. Den frühesten Zeitpunkt gibt <code>.min()</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>pro_tag = orders.groupby(___).size()
spitzentag = pro_tag.___()
erste = orders[&#x27;order_purchase_timestamp&#x27;].___()</pre>
</details>

**⭐ 2.6 · Bedeutung und Speicherung (ohne Code):** Tragen Sie für vier Spalten die Skala ein (nominal, ordinal,
Intervall, Verhältnis) und ob der Datentyp passt. Den Datentyp sehen Sie im Steckbrief. Doppelklick auf diese Zelle.

| Spalte | Tabelle | Skala | dtype | passt? |
|---|---|---|---|---|
| `order_status` | orders | … | … | … |
| `order_purchase_timestamp` | orders | … | … | … |
| `customer_zip_code_prefix` | kunden | … | … | … |
| `product_weight_g` | produkte | … | … | … |

Zum Staunen: Die Spaltennamen `product_name_lenght` und `product_description_lenght` sind kein Fehler von uns – der
Tippfehler steht so im Original.

## ⭐ 2.7 · Wo ist die Null geblieben?

Aus der Vorlesung: Die Postleitzahl 09790 wird als Zahl eingelesen – übrig bleibt 9790. Bei wie vielen Kunden?

In [ ]:
# ===== Jetzt Sie (⭐ 2.7): Anteil der Postleitzahlen mit nur 4 Ziffern =====
plz_text = kunden['customer_zip_code_prefix'].astype(str)     # fertig: die Zahl wieder als Text
anteil_4 = None          # TODO: Anteil der Einträge mit weniger als 5 Zeichen – .str.len(), Vergleich, .mean()

# --- Check ---
if anteil_4 is None:
    print('… noch offen')
elif abs(anteil_4 - 0.24129886063092687) > 1e-9:
    print('✗ prüfen: erwartet 24,1 % – (plz_text.str.len() < 5).mean()')
else:
    print(f'✓ richtig · {anteil_4:.1%} der Postleitzahlen haben ihre führende Null verloren. Reparatur: beim Einlesen '
          'dtype={\'customer_zip_code_prefix\': str} angeben oder nachträglich plz_text.str.zfill(5).')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>.str.len()</code> gibt je Eintrag die Zahl der Zeichen. Der Vergleich <code>&lt; 5</code> macht daraus True oder False, und <code>.mean()</code> ist der Anteil der True – wie in Woche 2 bei den Verspätungen.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>anteil_4 = (plz_text.str.___() &lt; 5).___()</pre>
</details>

---
# Teil 3 · Lieferzeiten: Was ist herausgekommen? (zu zweit, ca. 8 Minuten)

Frage 3 der Vorlesung: die Verteilung ansehen, mit Quantilen beschreiben, Ausreißer finden – und zwar mit einer
**eigenen Funktion** für die Regel von Tukey.

> 📘 Hängen Sie an der Schreibweise? Ein Klick öffnet das Grundlagen-Heft aus Woche 2 in einem neuen Tab an der richtigen Stelle: [Abschnitt 4 · Funktionen aufrufen](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-4) und [Abschnitt 7 · Zeilen filtern](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-7).

In [ ]:
# ===== Vorgemacht (3.1): die Lieferzeit in ganzen Tagen – wie in Woche 2 =====
lieferung = orders[(orders['order_status'] == 'delivered') & orders['order_delivered_customer_date'].notna()].copy()
lieferung['tage'] = (lieferung['order_delivered_customer_date'] - lieferung['order_purchase_timestamp']).dt.days
tage = lieferung['tage']
print(tage.describe().round(1))                          # count, mean, std, min, Quartile, max
print('90-%-Quantil:', tage.quantile(0.9), 'Tage')       # in 9 von 10 Fällen reicht das – das Lieferversprechen

## 3.2 · Jetzt Sie: Quartile und IQR der neun Bestellungen

Die neun Bestellungen nach Fortaleza von der Folie „Quartile von Hand“. Von Hand: Q1 an Platz 1 + ¼ · 8 = 3, Q3 an
Platz 7. Rechnen Sie nach, ob pandas dasselbe sagt.

In [ ]:
# ===== Jetzt Sie (3.2): Q1, Q3 und IQR =====
neun = pd.Series([15, 21, 23, 25, 31, 34, 36, 40, 168])   # fertig: Lieferzeiten in Tagen, sortiert
q1 = None        # TODO: das 25-%-Quantil
q3 = None        # TODO: das 75-%-Quantil
iqr = None       # TODO: Q3 minus Q1

# --- Check ---
if q1 is None or q3 is None or iqr is None:
    print('… noch offen')
elif (q1, q3) != (23, 36):
    print('✗ prüfen: erwartet Q1 = 23 und Q3 = 36 – neun.quantile(0.25) und neun.quantile(0.75)')
elif iqr != 13:
    print('✗ prüfen: erwartet IQR = 13 – q3 - q1')
else:
    print(f'✓ richtig · Q1 = {q1:.0f}, Q3 = {q3:.0f}, IQR = {iqr:.0f} Tage – wie von Hand. '
          f'Zäune: {q1 - 1.5 * iqr} und {q3 + 1.5 * iqr} Tage.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Wie <code>tage.quantile(0.9)</code> in 3.1, nur mit <code>neun</code> und 0.25 beziehungsweise 0.75.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>q1 = neun.quantile(___)
q3 = neun.quantile(___)
iqr = ___ - ___</pre>
</details>

## 3.3 · Jetzt Sie: die Regel von Tukey als Funktion

Ein Ausreißer-Kandidat liegt **unter Q1 − 1,5 · IQR** oder **über Q3 + 1,5 · IQR**. Schreiben Sie die beiden Zäune in
die Funktion. Dann prüft der Check sie zweimal: an den neun Bestellungen und an allen 96 470 Lieferungen.

In [ ]:
# ===== Jetzt Sie (3.3): die Funktion iqr_ausreisser =====
def iqr_ausreisser(s):
    """Gibt die Werte von s zurück, die außerhalb der Zäune von Tukey liegen."""
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    unten = None         # TODO: unterer Zaun
    oben = None          # TODO: oberer Zaun
    if unten is None or oben is None:
        return None      # (noch nicht fertig)
    return s[(s < unten) | (s > oben)]

# --- Check ---
test = iqr_ausreisser(neun)
if test is None:
    print('… noch offen')
elif list(test) != [168]:
    print(f'✗ prüfen: bei den neun Bestellungen liegt nur die 168 außerhalb (Zäune 3,5 und 55,5), gefunden: {list(test)}')
elif len(iqr_ausreisser(tage)) != 5022:
    print('✗ prüfen: bei allen Lieferungen erwartet 5 022 Kandidaten – stimmen beide Zäune?')
else:
    print(f'✓ richtig · neun Bestellungen: nur die 168 · alle {len(tage)} Lieferungen: {len(iqr_ausreisser(tage))} '
          'Kandidaten über 28,5 Tagen. Das sind 5 % – zu viele zum Löschen, genug zum Nachfragen.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Die Formel steht über der Zelle. In der Funktion heißen die Quartile <code>q1</code> und <code>q3</code>, der Abstand <code>iqr</code>. Malpunkt ist <code>*</code>, die Kommazahl schreiben Sie mit Punkt: <code>1.5</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>unten = q1 - ___ * iqr
oben = ___ + 1.5 * ___</pre>
</details>

## 3.4 · Boxplots vergleichen: São Paulo gegen Ceará (nur ausführen)

Der Bundesstaat steht in der Tabelle `kunden`. `merge` legt die Zeilen mit derselben `customer_id` nebeneinander –
wie in Woche 1.

In [ ]:
# ===== Vorgemacht (3.4): Boxplots je Bundesstaat =====
mit_staat = lieferung.merge(kunden, on='customer_id')                     # Bundesstaat dazu
zwei = mit_staat[mit_staat['customer_state'].isin(['SP', 'CE'])]
print(zwei.groupby('customer_state')['tage'].quantile([0.5, 0.9]).unstack().round(0))   # Median und 90 % je Staat
zwei.boxplot(column='tage', by='customer_state', figsize=(8, 4))
plt.ylim(0, 80)                                                           # Ausreißer über 80 Tage abgeschnitten
plt.suptitle('')
plt.title('Lieferzeit in Tagen')
plt.show()

## ⭐ 3.5 · Ausreißer nur in São Paulo

Ihre Funktion funktioniert mit jeder Spalte. Wie viele Kandidaten gibt es, wenn man nur São Paulo betrachtet?

In [ ]:
# ===== Jetzt Sie (⭐ 3.5): die Funktion für einen Bundesstaat =====
tage_sp = mit_staat.loc[mit_staat['customer_state'] == 'SP', 'tage']     # fertig: nur São Paulo
aus_sp = None        # TODO: Anzahl der Ausreißer-Kandidaten in tage_sp – Ihre Funktion, dann len()

# --- Check ---
if aus_sp is None:
    print('… noch offen')
elif aus_sp != 2021:
    print('✗ prüfen: erwartet 2 021 – len(iqr_ausreisser(tage_sp))')
else:
    print(f'✓ richtig · {aus_sp} Kandidaten in São Paulo. Dort liegen die Zäune viel tiefer: Was in Ceará normal ist, '
          'ist in São Paulo schon auffällig. Ausreißer hängen von der Gruppe ab.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Wie im Check von 3.3: <code>iqr_ausreisser(…)</code> gibt die auffälligen Werte zurück, <code>len(…)</code> zählt sie.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>aus_sp = len(iqr_ausreisser(___))</pre>
</details>

**3.6 Deuten (ein Satz):** Welches Lieferversprechen würden Sie für Ceará geben, welches für São Paulo – und warum nicht den Mittelwert?

*Ihre Antwort:*

---
# Teil 4 · Qualität: Wie gut ist die Messung? (zu zweit, ca. 8 Minuten – 4.7 nicht vergessen)

Frage 4 der Vorlesung: Je Dimension eine Prüfung, jede ist **eine Zeile Code**. Am Ende schreiben Sie die Befunde in
eine Tabelle – Ihr erster Qualitätsbericht.

> 📘 Hängen Sie an der Schreibweise? Ein Klick öffnet das Grundlagen-Heft aus Woche 2 in einem neuen Tab an der richtigen Stelle: [Abschnitt 3 · Vergleichen: True und False](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-3) und [Abschnitt 7 · Zeilen filtern](https://colab.research.google.com/github/ZhuosHub/Data_Science_HRW_Praktikum/blob/main/W2/W2_Praktikum_Grundlagen.ipynb#scrollTo=abschnitt-7).

In [ ]:
# ===== Vorgemacht (4.1): Vollständigkeit – fehlende Werte je Spalte und eine Plausibilitätsregel =====
print(orders.isna().sum())                                                 # leere Zellen je Spalte
zugestellt_ohne_datum = ((orders['order_status'] == 'delivered')
                         & orders['order_delivered_customer_date'].isna()).sum()
print('zugestellt, aber ohne Zustelldatum:', zugestellt_ohne_datum)

Die meisten Lücken sind **erklärbar**: Wer storniert, wird nie zugestellt. Verdächtig sind die 8 Bestellungen mit
Status *delivered*, aber ohne Datum.

## 4.2 · Jetzt Sie: Gültigkeit – ist der Wert überhaupt möglich?

In [ ]:
# ===== Jetzt Sie (4.2): Produkte mit einem Gewicht von 0 Gramm oder weniger =====
gewicht_0 = None         # TODO: Anzahl der Produkte mit product_weight_g <= 0

# --- Check ---
if gewicht_0 is None:
    print('… noch offen')
elif gewicht_0 != 4:
    print('✗ prüfen: erwartet 4 – (produkte[\'product_weight_g\'] <= 0).sum()')
else:
    print(f'✓ richtig · {gewicht_0} Produkte wiegen angeblich nichts. Ein Paket ohne Gewicht gibt es nicht.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Bedingung in Klammern, dann <code>.sum()</code>: Jedes True zählt als 1 – so wie in 4.1.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>gewicht_0 = (produkte[&#x27;product_weight_g&#x27;] ___ 0).___()</pre>
</details>

## ⭐ 4.3 · Jetzt Sie: Genauigkeit – zwei Spalten gegeneinander prüfen

Ein Wert kann möglich und trotzdem falsch sein. Oft merkt man es erst im Vergleich: Kann ein Paket an die Spedition
gehen, **bevor** die Zahlung freigegeben ist?

In [ ]:
# ===== Jetzt Sie (⭐ 4.3): an die Spedition vor der Freigabe =====
vor_freigabe = None      # TODO: Anzahl Bestellungen mit order_delivered_carrier_date < order_approved_at

# --- Check ---
if vor_freigabe is None:
    print('… noch offen')
elif vor_freigabe != 1359:
    print('✗ prüfen: erwartet 1 359 – (orders[\'order_delivered_carrier_date\'] < orders[\'order_approved_at\']).sum()')
else:
    print(f'✓ richtig · {vor_freigabe} Bestellungen – möglich (Vorkasse?), aber verdächtig. Eine Frage an Olist.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Zwei Spalten mit <code>&lt;</code> vergleichen gibt je Zeile True oder False. Dann <code>.sum()</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>vor_freigabe = (orders[&#x27;order_delivered_carrier_date&#x27;] ___ orders[&#x27;___&#x27;]).sum()</pre>
</details>

## ⭐ 4.4 · Jetzt Sie: Konsistenz – passt die Übersetzung?

Jede Kategorie soll in der Tabelle `uebersetzung` einen englischen Namen haben. Hat sie?

In [ ]:
# ===== Jetzt Sie (⭐ 4.4): Produkte, deren Kategorie keine Übersetzung hat =====
uebersetzt = uebersetzung['product_category_name']           # fertig: alle Kategorien mit englischem Namen
hat_kategorie = produkte['product_category_name'].notna()   # fertig: die 610 ohne Kategorie zählen hier nicht mit
ohne_uebersetzung = None   # TODO: Anzahl Produkte mit Kategorie, die nicht in uebersetzt steht – .isin(), ~ heißt „nicht“

# --- Check ---
if ohne_uebersetzung is None:
    print('… noch offen')
elif ohne_uebersetzung != 13:
    print('✗ prüfen: erwartet 13 – (hat_kategorie & ~produkte[\'product_category_name\'].isin(uebersetzt)).sum()')
else:
    welche = produkte.loc[hat_kategorie & ~produkte['product_category_name'].isin(uebersetzt), 'product_category_name']
    print(f'✓ richtig · {ohne_uebersetzung} Produkte, Kategorien: {sorted(welche.unique())}. '
          'Bei einer Verknüpfung (Woche 6) würden sie still verschwinden.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>spalte.isin(liste)</code> fragt je Zeile: Steht der Wert in der Liste? Das Zeichen <code>~</code> dreht True und False um, <code>&amp;</code> verbindet zwei Bedingungen. Dann <code>.sum()</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>ohne_uebersetzung = (hat_kategorie &amp; ~produkte[&#x27;product_category_name&#x27;].___(uebersetzt)).sum()</pre>
</details>

## 4.5 · Jetzt Sie: Eindeutigkeit – wer steht doppelt drin?

In [ ]:
# ===== Jetzt Sie (4.5): doppelte Menschen in der Tabelle kunden =====
print('komplett doppelte Zeilen in orders:', orders.duplicated().sum())          # fertig: keine
wiederholer = None       # TODO: Anzahl Zeilen in kunden, deren customer_unique_id schon weiter oben vorkam

# --- Check ---
if wiederholer is None:
    print('… noch offen')
elif wiederholer != 3345:
    print('✗ prüfen: erwartet 3 345 – kunden.duplicated(\'customer_unique_id\').sum()')
else:
    print(f'✓ richtig · {wiederholer} Zeilen gehören zu Menschen, die schon einmal bestellt haben (99 441 − 96 096). '
          'Kein Fehler – aber wer „Kunden“ zählt, muss sagen, welche Spalte er meint.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>.duplicated()</code> markiert Zeilen, die weiter oben schon vorkamen. In der Klammer können Sie sagen, welche Spalte zählen soll – hier <code>'customer_unique_id'</code>. Dann <code>.sum()</code>.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>wiederholer = kunden.duplicated(&#x27;___&#x27;).sum()</pre>
</details>

## ⭐ 4.6 · Jetzt Sie: Aktualität – wie alt ist die Messung?

In [ ]:
# ===== Jetzt Sie (⭐ 4.6): der späteste Bestellzeitpunkt =====
letzte = None            # TODO: der größte Wert in order_purchase_timestamp

# --- Check ---
if letzte is None:
    print('… noch offen')
elif str(letzte) != '2018-10-17 17:30:18':
    print('✗ prüfen: erwartet 2018-10-17 – orders[\'order_purchase_timestamp\'].max()')
else:
    print(f'✓ richtig · Die letzte Bestellung ist vom {letzte:%d.%m.%Y}. Für Fragen über heute sind die Daten acht '
          'Jahre alt – und die letzten Wochen sind unvollständig: Viele Bestellungen waren noch unterwegs.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p>Den größten Wert einer Spalte gibt <code>.max()</code> – bei Zeitpunkten ist das der späteste.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>letzte = orders[&#x27;order_purchase_timestamp&#x27;].___()</pre>
</details>

## 4.7 · Der Qualitätsbericht (ohne Code)

Tragen Sie Ihre Befunde ein – je Zeile: **was**, **wie viel**, **was es bedeutet**, **was wir tun**. Ein Beispiel steht
schon da (Doppelklick auf diese Zelle). Pflicht sind **Gültigkeit** (4.2) und **Eindeutigkeit** (4.5); die Zeilen mit
⭐ füllen Sie, wenn Sie die Aufgabe gemacht haben. Stichworte reichen – zwei Zeilen stellen Sie um 17:30 vor.

| Dimension | Befund: was, wie viel | was es bedeutet | was wir tun |
|---|---|---|---|
| Vollständigkeit | 610 Produkte ohne Kategorie, Namenslänge, Beschreibung und Fotozahl | nie vollständig angelegt | als „unbekannt“ führen, bei Olist nachfragen |
| Gültigkeit | … | … | … |
| Genauigkeit ⭐ | … | … | … |
| Konsistenz ⭐ | … | … | … |
| Eindeutigkeit | … | … | … |
| Aktualität ⭐ | … | … | … |

---
# 17:30 · Vorstellen: Ihr Qualitätsbericht

Jedes Team hat **zwei Minuten**. Lesen Sie aus 4.7 **zwei Zeilen** vor – je: was, wie viel, was es bedeutet, was Sie
tun – und nennen Sie **eine Zahl, die Sie heute überrascht hat** (zum Beispiel aus 2.5 oder 3.4). Die anderen hören
zu: Welche Dimension ist das? Würden wir dasselbe tun?

---
# Teil 5 · Ihr Projekt: der Qualitätsbericht (zu Hause, bis zum Zwischenstand am 03.12.)

Kopieren Sie die Funktion `steckbrief` aus 2.2 in Ihr Projekt-Notebook und stellen Sie Ihrem Datensatz die vier
Fragen. Der Bericht hat vier Blöcke – dieselben wie auf der Folie „Der Qualitätsbericht“:

| Block | Was hineingehört | Python, das hilft |
|---|---|---|
| 1 · Woran? | Quelle, Zeitraum, was eine Zeile ist, Anzahl Zeilen, Schlüssel | `df.shape` · `df['id'].is_unique` |
| 2 · Was? | je wichtiger Spalte: Bedeutung, Skala, Datentyp – passt er? | `steckbrief(df)` |
| 3 · Was kam heraus? | wichtige Kennzahlen, Quantile, Boxplots, Ausreißer | `describe()` · `quantile()` · `iqr_ausreisser()` |
| 4 · Wie gut? | Befunde je Dimension (Tabelle wie in 4.7), fehlende Werte und warum, Fragen an die Quelle | `isna()` · `duplicated()` · Vergleiche |

**Schluss:** Speichern Sie Ihre Kopie (Datei → Speichern). Die Musterlösung steht nach dem Praktikum in Moodle.
Nächste Woche reparieren wir, was wir heute gefunden haben: **Bereinigen und Transformieren**.

---
# Kür · Die Wetterstation selbst prüfen

Für alle, die mit Teil 1 bis 4 fertig sind: die Tageswerte der DWD-Station Essen-Bredeney aus der Vorlesung, direkt
vom Deutschen Wetterdienst. Die Zelle lädt ein ZIP-Archiv (knapp 1 MB) und liest die Datei mit den Messwerten.

*Falls das Laden scheitert:* Ohne Internetzugang zum DWD geht diese Kür nicht – dann einfach überspringen.

In [ ]:
# ===== Vorgemacht (Kür A): die Wetterdaten laden =====
import io, urllib.request, zipfile
DWD_URL = 'https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/tageswerte_KL_01303_19350101_20251231_hist.zip'
with urllib.request.urlopen(DWD_URL, timeout=60) as antwort:
    archiv = zipfile.ZipFile(io.BytesIO(antwort.read()))
name = [n for n in archiv.namelist() if n.startswith('produkt_')][0]          # die Datei mit den Messwerten
wetter = pd.read_csv(archiv.open(name), sep=';', skipinitialspace=True, encoding='latin-1')
wetter['datum'] = pd.to_datetime(wetter['MESS_DATUM'].astype(str), format='%Y%m%d')   # 19350101 → Datum
print(wetter.shape)
print('fehlende Werte laut isna():', wetter['SHK_TAG'].isna().sum())
print('mittlere Schneehöhe, einfach so gerechnet:', round(wetter['SHK_TAG'].mean(), 1), 'cm')

Minus 88 Zentimeter Schnee im Mittel – und `isna()` meldet keine einzige Lücke. Der Grund: Wo nichts gemessen wurde,
steht **−999**.

## Kür A1 · Jetzt Sie: den Platzhalter ersetzen

In [ ]:
# ===== Jetzt Sie (Kür A1): −999 ist kein Messwert =====
schnee = None            # TODO: die Spalte SHK_TAG, in der -999 durch np.nan ersetzt ist – .replace(alt, neu)

# --- Check ---
if schnee is None:
    print('… noch offen')
elif schnee.isna().sum() != 2819:
    print('✗ prüfen: erwartet 2 819 fehlende Werte – wetter[\'SHK_TAG\'].replace(-999, np.nan)')
else:
    print(f'✓ richtig · {schnee.isna().sum()} Tage ohne Messwert, mittlere Schneehöhe {schnee.mean():.1f} cm statt '
          '−88,4 cm. Das klingt schon eher nach Ruhrgebiet.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>.replace(-999, np.nan)</code> ersetzt jeden Eintrag −999 durch einen echten fehlenden Wert. Danach lässt <code>.mean()</code> ihn weg.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>schnee = wetter[&#x27;SHK_TAG&#x27;].replace(___, np.nan)</pre>
</details>

## Kür A2 · Jetzt Sie: Lücken ohne Loch

Welche Tage fehlen ganz? `pd.date_range(erster, letzter)` erzeugt **alle** Tage dazwischen; `.difference(…)` lässt die
übrig, für die es keine Zeile gibt.

In [ ]:
# ===== Jetzt Sie (Kür A2): Tage ohne Zeile =====
alle_tage = pd.date_range(wetter['datum'].min(), wetter['datum'].max())     # fertig: jeder Tag von 1935 bis 2025
fehlende_tage = None     # TODO: die Tage aus alle_tage, die in wetter['datum'] nicht vorkommen – .difference(…)

# --- Check ---
if fehlende_tage is None:
    print('… noch offen')
elif len(fehlende_tage) != 1460:
    print('✗ prüfen: erwartet 1 460 Tage – alle_tage.difference(wetter[\'datum\'])')
else:
    jahre = sorted(set(fehlende_tage.year))
    print(f'✓ richtig · {len(fehlende_tage)} Tage ohne Zeile, in den Jahren {jahre}. Nirgends eine leere Zelle – '
          'die Tage fehlen einfach. Nur die mitgelieferten Metadaten nennen sie.')

<details><summary><b>💡 Tipp</b> · erst selbst versuchen, dann aufklappen</summary>
<p><code>alle_tage.difference(wetter['datum'])</code> gibt die Tage zurück, die in <code>alle_tage</code> stehen, aber nicht in der Spalte.</p>
</details>
<details><summary><b>🧩 Gerüst</b> · nur, wenn der Tipp nicht reicht: die Zeilen mit Lücken ___</summary>
<pre>fehlende_tage = alle_tage.___(wetter[&#x27;datum&#x27;])</pre>
</details>